# Micro Proyecto 2: Modelos avanzados para el Procesamiento de Lenguaje Natural

**Objetivo:** Creación de un modelo GPT -0 y Fine-tunning con base en Shakespeare.

# **1. GPT desde 0**

## Instalación de librerías

In [1]:
!pip install transformers datasets accelerate -q

import torch
import torch.nn as nn
from torch.nn import functional as F

## Hiperparámetros a utilizar

In [2]:
batch_size = 32
block_size = 128
max_iters = 2000
eval_interval = 200
learning_rate = 3e-4
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 100
n_embd = 128
n_head = 4
n_layer = 4
dropout = 0.2
print(device)

cuda


## Import de datos y verificación del dataset

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [9]:
import os
dataset_path = '/content/drive/MyDrive/Andes/202612/NLP2/Microproyecto2'
os.makedirs(dataset_path, exist_ok=True)
os.chdir(dataset_path)
os.getcwd()

'/content/drive/MyDrive/Andes/202612/NLP2/Microproyecto2'

In [13]:
with open('input-2-.txt', 'r', encoding='utf-8') as f:
    text = f.read()

chars = sorted(list(set(text)))
vocab_size = len(chars)
print(chars)
print(vocab_size)

['\n', ' ', '!', '$', '&', "'", ',', '-', '.', '3', ':', ';', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']
65


## Split del dataset

In [14]:
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])

# Split
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

In [16]:
print(len(val_data))#111540
print(len(train_data)) #1003854

111540
1003854


In [17]:
def get_batch(split):
    d = train_data if split == 'train' else val_data
    ix = torch.randint(len(d) - block_size, (batch_size,))
    x = torch.stack([d[i:i+block_size] for i in ix])
    y = torch.stack([d[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

## Modelo GPT

In [18]:
class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)
        q = self.query(x)
        wei = q @ k.transpose(-2, -1) * C**-0.5
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)
        v = self.value(x)
        out = wei @ v
        return out

class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out

class FeedForward(nn.Module):
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedForward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

class GPTLanguageModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok_emb = self.token_embedding_table(idx)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device))
        x = tok_emb + pos_emb
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]
            logits, loss = self(idx_cond)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

## Instancia y loop de train

In [19]:
model_scratch = GPTLanguageModel().to(device)
optimizer = torch.optim.AdamW(model_scratch.parameters(), lr=learning_rate)
## loop
for iter in range(max_iters):
    if iter % eval_interval == 0 or iter == max_iters - 1:
        model_scratch.eval()
        losses = {'train': 0, 'val': 0}
        with torch.no_grad():
            for split in ['train', 'val']:
                losses_arr = torch.zeros(eval_iters)
                for k in range(eval_iters):
                    X, Y = get_batch(split)
                    logits, loss = model_scratch(X, Y)
                    losses_arr[k] = loss.item()
                losses[split] = losses_arr.mean()
        print(f"Paso {iter}: loss entrenamiento {losses['train']:.4f}, loss validación {losses['val']:.4f}")
        model_scratch.train()

    xb, yb = get_batch('train')
    logits, loss = model_scratch(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

Paso 0: loss entrenamiento 4.3813, loss validación 4.3814
Paso 200: loss entrenamiento 2.5384, loss validación 2.5390
Paso 400: loss entrenamiento 2.4617, loss validación 2.4628
Paso 600: loss entrenamiento 2.3945, loss validación 2.4097
Paso 800: loss entrenamiento 2.3126, loss validación 2.3384
Paso 1000: loss entrenamiento 2.2266, loss validación 2.2528
Paso 1200: loss entrenamiento 2.1476, loss validación 2.1823
Paso 1400: loss entrenamiento 2.0826, loss validación 2.1298
Paso 1600: loss entrenamiento 2.0311, loss validación 2.0837
Paso 1800: loss entrenamiento 1.9674, loss validación 2.0425
Paso 1999: loss entrenamiento 1.9239, loss validación 2.0052


In [20]:
## Save del modelo
torch.save(model_scratch.state_dict(), 'gpt_desde_cero_checkpoint.pth')

##Generación de 3 ejemplos

In [21]:
model_scratch.eval()
for i in range(3):
    context = torch.zeros((1, 1), dtype=torch.long, device=device)
    generated_text = decode(model_scratch.generate(context, max_new_tokens=150)[0].tolist())
    print(f"\n--- Ejemplo {i+1} del GPT desde 0 ---")
    print(generated_text)


--- Ejemplo 1 del GPT desde 0 ---

ERNENTIUS:
All land gret, thet-ak that for'ss?

IS ROMESY IURIO:
Moigh and gover for uld of her drost then.
The Thy grighty dept Hice thergmite;
And p

--- Ejemplo 2 del GPT desde 0 ---

Selosss, at a dom theeis, I'd theehs by this sings.

CLANPEOWARLIUS:
Consill sopon, gestaram.

KING RICKI:
Thath hee neests, now of pret ment tis in o

--- Ejemplo 3 del GPT desde 0 ---

is her apwith wor a's yit hord for.

CEMPO:
Cod, mye shoule, fran of the parlivest:
The be thon goo to che thout nort thice? myers
He some hy brok, su


#**2. Fine-tunning de GPT - 2**

## Import de librerías

In [24]:
from transformers import GPT2Tokenizer, GPT2LMHeadModel, Trainer, TrainingArguments, DataCollatorForLanguageModeling, pipeline
from datasets import Dataset

tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
model_gpt2 = GPT2LMHeadModel.from_pretrained('gpt2').to(device)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## Carga de datos

In [29]:
tokenizer.pad_token = tokenizer.eos_token

tokenized_text = tokenizer(text, return_attention_mask=False)

raw_datasets = Dataset.from_dict({"text": [tokenized_text["input_ids"]]})

In [30]:
# Función de agrupación de texto
def group_texts(examples):
    concatenated_examples = {k: sum(examples[k], []) for k in examples.keys()}
    total_length = len(concatenated_examples[list(examples.keys())[0]])
    total_length = (total_length // block_size) * block_size
    result = {
        k: [t[i : i + block_size] for i in range(0, total_length, block_size)]
        for k, t in concatenated_examples.items()
    }
    result["labels"] = result["text"].copy()
    return result

#INstancia de dataset
processed_datasets = raw_datasets.map(
    group_texts,
    batched=True,
    num_proc=4,
    remove_columns=raw_datasets.column_names,
)

num_proc must be <= 1. Reducing num_proc to 1 for dataset of size 1.


Map:   0%|          | 0/1 [00:00<?, ? examples/s]

In [38]:
# Split del dataset
processed_datasets = processed_datasets.rename_column("text", "input_ids") # Rename 'text' to 'input_ids'
train_test_split = processed_datasets.train_test_split(test_size=0.2) # 20%
train_dataset = train_test_split['train']
val_dataset = train_test_split['test']

print(f"Training dataset size: {len(train_dataset)}")
print(f"Validation dataset size: {len(val_dataset)}")

Training dataset size: 2112
Validation dataset size: 528


## Trainer

In [45]:
training_args = TrainingArguments(
    output_dir='./gpt2_finetuned_checkpoint',
    num_train_epochs=5, # Increased epochs to 5
    per_device_train_batch_size=16,
    save_steps=500,
    save_total_limit=2,
    logging_steps=100,
    report_to="none"
)

trainer = Trainer(
    model=model_gpt2,
    args=training_args,
    data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False),
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
)

## Train

In [46]:
trainer.train()

Step,Training Loss


Step,Training Loss
100,3.217903
200,3.171046
300,3.138426
400,3.123513
500,3.073398
600,3.033404


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=660, training_loss=3.1203458843809186, metrics={'train_runtime': 428.4479, 'train_samples_per_second': 24.647, 'train_steps_per_second': 1.54, 'total_flos': 689810964480000.0, 'train_loss': 3.1203458843809186, 'epoch': 5.0})

In [47]:
trainer.save_model('./gpt2_finetuned_final')
tokenizer.save_pretrained('./gpt2_finetuned_final')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./gpt2_finetuned_final/tokenizer_config.json',
 './gpt2_finetuned_final/tokenizer.json')

## Generación de 3 ejemplos

In [48]:
generator = pipeline('text-generation', model='./gpt2_finetuned_final', tokenizer='./gpt2_finetuned_final', device=0 if torch.cuda.is_available() else -1)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

In [49]:
for i in range(3):
    prompt = "CORIOLANUS:\n"
    result = generator(prompt, max_length=100, num_return_sequences=1, truncation=True)
    print(f"\n--- Ejemplo {i+1} (GPT-2) ---")
    print(result[0]['generated_text'])

Passing `generation_config` together with generation-related arguments=({'num_return_sequences', 'max_length'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=256) and `max_length`(=100) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=100) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



--- Ejemplo 1 (GPT-2) ---
CORIOLANUS:
Amen.

AUFIDIUS:
Sir, you are the sovereign; you are both:
And, for that, he is your sovereign.

SICINIUS:
All in all, he is, sir:

AUFIDIUS:
Go, go, go, go; you are my sovereign,
And you--

CORIOLANUS:
You all are my citizens, sir;
And I, in my sovereign's presence,
Have appointed your consulship.

SICINIUS:

AUFIDIUS:
If any of you
Commit your faults to your sovereign,
You shall be my consul.

CORIOLANUS:
I have no part in it, sir; if you
Have, I'll send you to your sovereign's assistance.

LUCIO:
But, I am a man, sir, and I will not yield to your will.

AUFIDIUS:
You shall not be my sovereign, sir.

CORIOLANUS:
Your sovereign's help, sir, shall not be your sovereign's help.

LUCIO


Both `max_new_tokens` (=256) and `max_length`(=100) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



--- Ejemplo 2 (GPT-2) ---
CORIOLANUS:
That, indeed, thou wouldst like me to be.

CORIOLANUS:
Then tell me who thou art:
I'll go along with thee to-morrow morning.

MENENIUS:

CORIOLANUS:

CORIOLANUS:
By, by, by:
Pray, what is thy name?

MENENIUS:
Coriolanus, Coriolanus.

CORIOLANUS:
I shall name him after me:
A friend, a cousin, a friend
Of mine, a man that thou hast love's ears to hear:
I'll be his friend for a while, and then thou shalt know
What's in him.

MENENIUS:

CORIOLANUS:
He shall be your friend, a friend, a friend.

CORIOLANUS:
A friend, a friend.

MENENIUS:
I have thought of him, and am as sure
That he will be your friend as I am his friend.

CORIOLANUS:
My dear friends, I am sure of it,
And I do believe thee.

--- Ejemplo 3 (GPT-2) ---
CORIOLANUS:
Let him go away, and make his way.

CORIOLANUS:
Why, he's not gone:
He's not left:
I do not know what it is that makes him leave
But I know that it is that makes him leave.

KATHARINA:
Is he gone?

CORIOLANUS:
No, I do not know;